# Lab 6 · PureSVD

The runnable companion of the lab page, `docs/labs/06-puresvd.md` (also on the docs site). The page explains each step,
asks the questions and holds the hints and solutions; this notebook has the code.

- Run the setup cell first, every time the kernel starts (after a crash: **Restart**, then setup again).
- Cells that fit models take seconds to minutes on MovieLens. Run them where you train: on the rented box (VS Code's
  Remote - SSH extension opens this notebook there), or on the laptop if you allow training there.
- Cells that only read stored results (the baseline, `lab.compare`, `lab.segments`) are fine anywhere, once the box's
  results are fetched (`./scripts/fetch_lab_results.sh`).
- Save the notebook **without outputs** before you commit (the **...** menu → Clear All Outputs).

In [ ]:
from recbench import lab

lab.setup()
DATASET = "movielens-25m"
METHOD = "puresvd"

## Your starting point

The lab baseline on every dataset (stored results; nothing is trained here).

In [ ]:
lab.baseline(METHOD)

## Level 1: reproduce and read

### 1.1 Reproduce the baseline's best setting

Fit the baseline's best setting on the validation fold, in this notebook. The NDCG@10 must equal the baseline's validation score.

In [ ]:
best = lab.baseline(METHOD, DATASET)["best_settings"]
data, split = lab.load(DATASET)
model = lab.fit(METHOD, data, **best)
result = lab.evaluate(model, data, split)
print(result)
print("the baseline's best trial:", lab.baseline(METHOD, DATASET)["validation"])

### 1.2 Read the code

The questions are on the lab page; answer them with the code open next to this notebook.

### 1.3 Look inside

In [ ]:
import numpy as np

v = model.v  # items x directions
print("correlation of direction 1 with popularity:", round(float(np.corrcoef(np.abs(v[1:, 0]), data.item_pop[1:])[0, 1]), 3))
order = np.argsort(v[:, 1])
print("one end of direction 2:", [data.item_text[i] or data.item_ids[i] for i in order[:5]])
print("the other end:", [data.item_text[i] or data.item_ids[i] for i in order[-5:]])

**Exercise.** Name direction 3: list the five items at each of its two ends. What separates them?

<details><summary>Hint</summary>

Same as direction 2, with `v[:, 2]`.

</details>

<details><summary>Solution</summary>

```python
order = np.argsort(v[:, 2])
print([data.item_text[i] or data.item_ids[i] for i in order[:5]])
print([data.item_text[i] or data.item_ids[i] for i in order[-5:]])
```
There is no right answer: the SVD found the contrast from co-occurrence alone. Genre, age and audience are common readings.

</details>

In [ ]:
# Your code here

## Level 2: understand each setting

For each sweep: **write your prediction in the cell below it first**, then run the sweep. Every other setting stays at the baseline's best. The lab page explains what to look for.

In [ ]:
table = lab.sweep(METHOD, DATASET, "svd_factors", [16, 64, 256, 1024], start="best")
lab.plot_sweep(table)
table

**Your prediction for `svd_factors`, and what you saw:** ...

## Level 3: data tricks

Experiments run on the rented box over all five datasets, for example:

```bash
./scripts/run_lab_box.sh puresvd:no-time-knobs
```

Then fetch the results (`./scripts/fetch_lab_results.sh <host>`) and compare here. `no-time-knobs` is ready to run; `user-norm`
needs a small code change first (the lab page has the hint and the folded solution). After writing it, run its test:
`pytest -q tests/test_puresvd_variant.py`.

In [ ]:
lab.compare(METHOD, f"{METHOD}:no-time-knobs")

In [ ]:
lab.compare(METHOD, f"{METHOD}:user-norm")

## Level 4: one change from the literature

Write the variant (the lab page explains the idea, with hints and the folded solution), test it, uncomment
`eigenrec` in `labs/06-puresvd/experiments.yaml`, run it on the box, fetch, then compare and look at the segments.

In [ ]:
lab.compare(METHOD, f"{METHOD}:eigenrec")

In [ ]:
parts = lab.segments(METHOD, f"{METHOD}:eigenrec", DATASET)
lab.plot_segments(parts, "activity")

In [ ]:
parts["items"]

## Record your results

Fill in the table at the end of the lab page, then rebuild the scoreboard: `python -m recbench.lab scoreboard --docs`.